This function is meant to test NumPy @ operator used for dot product operations betwen higher order tensors to check if it is returning correct values. Some hard coded arrays will be used to evaluate sub-coefficient matrices, as done in _elemMatComput function in solver.py

In [1]:
import numpy as np
import sys
from pathlib import Path

# Starting from Model/Testing/, go up one level to Model/
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
from gen.gen_utilities import permutation_symbol

# variables useful for testing the element matrix calculation specifically
perm = permutation_symbol()
dphi_oE = np.array([[0.5372370636309108,-0.37960220374900183,1.7430044800653222],
                    [-0.028368137739709975,0.03302847357484874,0.2649110079584831],
                    [0.7943329089942659,-1.6325697578546674,-2.2614134476849013]], dtype=np.float64)

forceGP = np.array([[13278450720.390465,543622257715.53015,177153662843.55847],
                    [-57528419714.30026,22890227916.87219,-4147645793.9639397],
                    [-20272944568.310963,-235797398959.89944,151404811980.01657]], dtype=np.float64)

momentGP = np.array([[12.104458310772278,13.537659765346003,15.600363073384393],
                     [7.59765068265847,16.075827672753487,12.012157956312475],
                     [13.818987820248426,16.397121205372276,10.340404125119495]], dtype=np.float64)

matModF = np.diag([298572800.0, 298572800.0, 2299017])
matModM = np.diag([1.9999999992, 1.9999999992, 1.5238088])

In [3]:
# Test 1: Vector-vector dot product
v1 = np.array([1, 2, 3])
v2 = np.array([4, 5, 6])
result = v1 @ v2
expected = 32  # 1*4 + 2*5 + 3*6
print('result=',result)

result= 32


In [4]:
# Test 2: Standard matrix-matrix multiplication
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
result = A @ B
expected = np.array([[19, 22], [43, 50]])
print('result=',result)

result= [[19 22]
 [43 50]]


In [5]:
# Test 3: comparison between np.dot, @, einsum and matmul. All tests non square arrays.

A = np.random.randn(4, 3)
B = np.random.randn(3, 5)
resultOperator = A @ B
resultMatmul = np.matmul(A, B)
resultDot = np.dot(A, B) # only for 2D arrays
resultEinsum = np.einsum('ij,jk->ik',A,B)

# print('resultOperator=',resultOperator)
# print('resultMatmul=',resultMatmul)
# print('resultDot=',resultDot)
# print('resultEinSum=',resultEinsum)
if np.allclose(resultOperator, resultMatmul):
    print('All values are close between @ and np.matmul')
if np.allclose(resultOperator, resultDot):
    print('All values are close between @ and np.dot')
if np.allclose(resultOperator, resultEinsum):
    print('All values are close between @ and np.einsum')

All values are close between @ and np.matmul
All values are close between @ and np.dot
All values are close between @ and np.einsum


In [6]:
# Test 4: Matrix-vector multiplication

A = np.array([[1, 2], [3, 4]])
v = np.array([5, 6])
result = A @ v

expected = np.array([17, 39])
print('result=',result)

result= [17 39]


In [7]:
# Test 5: Vector-Matrix multiplication

A = np.array([[1, 2], [3, 4]])
v = np.array([5, 6])
result = v @ A

expected = np.array([23, 34])
print('result=',result)

result= [23 34]


In [8]:
# Test 6: Multiplication with identity matrix
A = np.array([[1, 2, 3],[4,5,6],[7,8,9]])
I = np.eye(3)
result = A @ I

print('result=',result)

result= [[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]


In [9]:
# Test 7: Test value from K12 matrix. 
# 12 refers to notation from Simo's work. 1- translation, 2-rotation.
i=2 # can be change to 0,1,2 to access particular predefined numpy vector 
arrDumO = np.zeros(shape=3,dtype=np.float64)
for a in range(3):
    for b in range(3):
        resultOp = dphi_oE[i] @ perm[:,b,:] @ matModF[:,a] - forceGP[i] @ perm[:,b,a]
        # manual calculation - refers to using for loop to compute dot products
        arrDumO[:] = 0.0
        dummyA = 0.0
        dummyB = 0.0
        for rr in range(3):
            dummyB += forceGP[i,rr] * perm[rr,b,a]
            for ll in range(3):
                dummyA += dphi_oE[i,rr] * perm[rr,b,ll] * matModF[ll,a]

        resultManual = dummyA - dummyB
        print('For a=',a,' b=',b)
        if np.isclose(resultOp, resultManual):
            print('Calculation is close resulOp=',resultOp,' resulManual=',resultManual,'\n') 
        else:
            print('Calculation is not close resulOp=',resultOp,' resulManual=',resultManual,'\n') 


For a= 0  b= 0
Calculation is close resulOp= 0.0  resulManual= 0.0 

For a= 0  b= 1
Calculation is close resulOp= 152080008525.0495  resulManual= 152080008525.0495 

For a= 0  b= 2
Calculation is close resulOp= 235309958036.10144  resulManual= 235309958036.10144 

For a= 1  b= 0
Calculation is close resulOp= -152080008525.0495  resulManual= -152080008525.0495 

For a= 1  b= 1
Calculation is close resulOp= 0.0  resulManual= 0.0 

For a= 1  b= 2
Calculation is close resulOp= -20510110769.081528  resulManual= -20510110769.081528 

For a= 2  b= 0
Calculation is close resulOp= -235793645654.27246  resulManual= -235793645654.27246 

For a= 2  b= 1
Calculation is close resulOp= 20274770753.1724  resulManual= 20274770753.1724 

For a= 2  b= 2
Calculation is close resulOp= 0.0  resulManual= 0.0 



In [10]:
# Test 8: Test value from K21 matrix. 
# 21 refers to notation from Simo's work. 2-rotation, 1- translation
i=0 # can be change to 0,1,2 to access particular predefined numpy vector 
for a in range(3):
    for b in range(3):
        resultOp = dphi_oE[i] @ perm[:,a,:] @ matModF[:,b] - forceGP[i] @ perm[:,a,b]
        # manual calculation - refers to using for loop to compute dot products
        dummyA = 0.0
        dummyB = 0.0
        for pp in range(3):
            dummyB += forceGP[i,pp] * perm[pp,a,b]
            for qq in range(3):
                dummyA += dphi_oE[i,pp] * perm[pp,a,qq] * matModF[qq,b]

        resultManual = dummyA - dummyB
        print('For a=',a,' b=',b)
        if np.isclose(resultOp, resultManual):
            print('Calculation is close resulOp=',resultOp,' resulManual=',resultManual,'\n') 


For a= 0  b= 0
Calculation is close resulOp= 0.0  resulManual= 0.0 

For a= 0  b= 1
Calculation is close resulOp= -176633249115.53284  resulManual= -176633249115.53284 

For a= 0  b= 2
Calculation is close resulOp= 543623130427.4498  resulManual= 543623130427.4498 

For a= 1  b= 0
Calculation is close resulOp= 176633249115.53284  resulManual= 176633249115.53284 

For a= 1  b= 1
Calculation is close resulOp= 0.0  resulManual= 0.0 

For a= 1  b= 2
Calculation is close resulOp= -13277215603.248148  resulManual= -13277215603.248148 

For a= 2  b= 0
Calculation is close resulOp= -543735596608.38965  resulManual= -543735596608.38965 

For a= 2  b= 1
Calculation is close resulOp= 13118046346.038406  resulManual= 13118046346.038406 

For a= 2  b= 2
Calculation is close resulOp= 0.0  resulManual= 0.0 



In [11]:
# Test 9: Test value from K2210 matrix. 
# 22 refers to notation from Simo's work. 2-rotation. 
# 10 means outerproduct of first derivation and zeroth derivative of interpolants.
i=2 # can be changed to 0,1,2 to access particular predefined numpy vector 
for a in range(3):
    for b in range(3):
        resultOp = momentGP[i] @ perm[:,a,b]
        # manual calculation - refers to using for loop to compute dot products
        dummy= 0.0
        for yy in range(3):
            dummy += momentGP[i,yy] * perm[yy,a,b]

        resultManual = dummy
        print('For a=',a,' b=',b)
        if np.isclose(resultOp, resultManual):
            print('Calculation is close resulOp=',resultOp,' resulManual=',resultManual,'\n') 

For a= 0  b= 0
Calculation is close resulOp= 0.0  resulManual= 0.0 

For a= 0  b= 1
Calculation is close resulOp= 10.340404125119495  resulManual= 10.340404125119495 

For a= 0  b= 2
Calculation is close resulOp= -16.397121205372276  resulManual= -16.397121205372276 

For a= 1  b= 0
Calculation is close resulOp= -10.340404125119495  resulManual= -10.340404125119495 

For a= 1  b= 1
Calculation is close resulOp= 0.0  resulManual= 0.0 

For a= 1  b= 2
Calculation is close resulOp= 13.818987820248426  resulManual= 13.818987820248426 

For a= 2  b= 0
Calculation is close resulOp= 16.397121205372276  resulManual= 16.397121205372276 

For a= 2  b= 1
Calculation is close resulOp= -13.818987820248426  resulManual= -13.818987820248426 

For a= 2  b= 2
Calculation is close resulOp= 0.0  resulManual= 0.0 



In [12]:
# Test 9: Test value from K2200a matrix. 
# 22 refers to notation from Simo's work. 2-rotation. 
# 00 means outerproduct of zeroth derivative of interpolants. a means first term in the equation.
i=0 # can be change to 0,1,2 to access particular predefined numpy vector 
for a in range(3):
    for b in range(3):
        resultOp = -dphi_oE[i] @ perm[:,a,:] @ matModF @ perm[:,b,:] @ dphi_oE[i]
        # manual calculation - refers to using for loop to compute dot products
        dummyA = 0.0
        for ll in range(3):
            for pp in range(3):
                for qq in range(3):
                    for ss in range(3):
                        dummyA += -dphi_oE[i,ll] * perm[ll,a,pp] * matModF[pp,qq] * perm[qq,b,ss] * dphi_oE[i,ss]

        resultManual = dummyA
        print('For a=',a,' b=',b)
        if np.isclose(resultOp, resultManual):
            print('Calculation is close resulOp=',resultOp,' resulManual=',resultManual,'\n') 

For a= 0  b= 0
Calculation is close resulOp= 907414742.8041391  resulManual= 907414742.8041391 

For a= 0  b= 1
Calculation is close resulOp= 468853.1891119099  resulManual= 468853.1891119099 

For a= 0  b= 2
Calculation is close resulOp= -279585543.1177143  resulManual= -279585543.1177143 

For a= 1  b= 0
Calculation is close resulOp= 468853.18911190995  resulManual= 468853.18911190995 

For a= 1  b= 1
Calculation is close resulOp= 907747010.1429784  resulManual= 907747010.1429784 

For a= 1  b= 2
Calculation is close resulOp= 197550198.01976943  resulManual= 197550198.01976943 

For a= 2  b= 0
Calculation is close resulOp= -279585543.1177143  resulManual= -279585543.1177143 

For a= 2  b= 1
Calculation is close resulOp= 197550198.01976943  resulManual= 197550198.01976943 

For a= 2  b= 2
Calculation is close resulOp= 129198868.57039566  resulManual= 129198868.57039568 



In [13]:
# Test 10: Test value from K2200a matrix. 
# 22 refers to notation from Simo's work. 2-rotation. 
# 00 means outerproduct of zeroth derivative of interpolants. a means first term in the equation.
i=0 # can be change to 0,1,2 to access particular predefined numpy vector 
for a in range(3):
    for b in range(3):
        resultOp = dphi_oE[i] @ perm[:,a,:] @ perm[:,b,:] @ forceGP[i]
        # manual calculation - refers to using for loop to compute dot products
        dummyA = 0.0
        for ss in range(3):
            for ll in range(3):
                for pp in range(3):
                    dummyA += dphi_oE[i,ss] * perm[ss,a,ll] * perm[ll,b,pp] * forceGP[i,pp]

        resultManual = dummyA
        print('For a=',a,' b=',b)
        if np.isclose(resultOp, resultManual):
            print('Calculation is close resulOp=',resultOp,' resulManual=',resultManual,'\n') 

For a= 0  b= 0
Calculation is close resulOp= -102419420960.48096  resulManual= -102419420960.48096 

For a= 0  b= 1
Calculation is close resulOp= -5040529155.832742  resulManual= -5040529155.832742 

For a= 0  b= 2
Calculation is close resulOp= 23144399093.967186  resulManual= 23144399093.967186 

For a= 1  b= 0
Calculation is close resulOp= 292054025459.4977  resulManual= 292054025459.4977 

For a= 1  b= 1
Calculation is close resulOp= -315913303870.89435  resulManual= -315913303870.89435 

For a= 1  b= 2
Calculation is close resulOp= 947536030661.3942  resulManual= 947536030661.3942 

For a= 2  b= 0
Calculation is close resulOp= 95173513637.53375  resulManual= 95173513637.53375 

For a= 2  b= 1
Calculation is close resulOp= -67247920817.62246  resulManual= -67247920817.62246 

For a= 2  b= 2
Calculation is close resulOp= 199226531161.23273  resulManual= 199226531161.23273 

